## Conditional residual risk

Residual risk estimates the probability of a terminal individual-possession loss **given that the player successfully completes the current one-second carry**. The model evaluates the state at the beginning of that carry; the subsequent observed trajectory is not an input. A successful carry does not determine how the individual possession will ultimately end, so the prediction concerns the remaining sequence of individual control.

Let $Y=1$ denote a terminal individual-possession loss and $Y=0$ a terminal outcome without loss, according to the conventions specified in the [residual-risk benchmark](../../benchmarks/05a_residual_risk_benchmark.qmd). Using the same state notation as Action Selection, let $s$ denote the current game state and $A$ the selected action. For this state, the conditional target is

$$
c(s)=P(Y=1\mid s,\,A=\mathrm{carry},\,\text{successful one-second carry}).
$$

Consequently, the probability of a terminal outcome **without loss**, under the same condition, is $1-c(s)$. This is a prediction about the current player's individual possession, not the later outcome of a teammate's control.

## Binary classification model

We estimate this probability with a regularized XGBoost classifier [@chen2016xgboost]. For feature vector $z_i$ and binary terminal label $y_i\in\{0,1\}$, the tree ensemble produces a score $f_\theta(z_i)$ and the logistic residual-loss probability

$$
\widehat c_i=\frac{1}{1+\exp[-f_\theta(z_i)]}.
$$

The parameters are fitted by minimizing the Bernoulli negative log-likelihood with the usual tree-complexity penalty,

$$
\mathcal L(\theta)=-\sum_i\left[y_i\log\widehat c_i+(1-y_i)\log(1-\widehat c_i)\right]+\Omega(\theta).
$$

Training observations are states preceding **completed, successful one-second carries**, with the label taken from the eventual terminal outcome of the same individual possession. Cohort verification, outcome conventions, model selection and held-out evaluation are documented in the [benchmark](../../benchmarks/05a_residual_risk_benchmark.qmd).

## Model inputs

Residual risk uses the same state-based features and construction procedures described in the [Action Selection vignette](../04_action_selection/04_action_selection.ipynb), with two changes. First, it adds $1-\pi(s)$, where $\pi(s)=P(A=\mathrm{pass}\mid s)$ is the pass-choice probability defined in Action Selection. Thus the additional input is the complementary probability of selecting a carry, not a separately fitted prediction. This is distinct from the carry-execution success probability $D(s)$, already included among the shared features. Second, it replaces the count of completed one-second carries with **elapsed time since the beginning of the individual possession**, measured at the prediction frame. Neither the remaining possession duration nor future tracking information is provided. These changes yield **31 conceptual inputs**, including the two categorical context variables.


## Feature importance and interpretation

The figure reports XGBoost **total split gain**, normalized within each fitted ensemble. One-hot encoded levels are aggregated into their original conceptual features. Bars represent the installed full-data model, while the five outer-fold estimates indicate variability across fitted models; their ranges are descriptive, not confidence intervals.

![Feature importance of the residual-risk model.](../../figures/feature_importance_residual_risk.png)

**Accessible space and the configuration of subsequent passing options are prominent predictors.** BCOS accounts for **14.21%** of production split gain and ranks between first and fourth across the five outer folds. This suggests that the carrier's available space provides information about terminal individual-possession loss even after conditioning on a successful one-second carry.

The number of opponents ahead of likely completed-pass destinations contributes **10.30%** and ranks first or second across folds. The model therefore relies not only on immediate conditions around the carrier but also on the spatial configuration of plausible subsequent passes. The estimated probability of successful one-second carrying (**8.83%**) and mean pass success (**8.01%**) are also prominent, linking the forecast to immediate execution conditions and available alternatives. However, the rank of mean pass success varies substantially across folds, so its production importance is less stable.

Team-line position under failed-pass outcomes (**5.60%**) and own-team xT under successful-pass outcomes (**5.24%**) further indicate that the model uses the tactical structure of potential continuations. Overall, these rankings are consistent with residual risk depending on both the carrier's freedom of movement and the quality of available options; they do not identify causal mechanisms.

Split gain is not a share of predictive accuracy or a causal effect. Correlated inputs can substitute for one another, and features with more potential split points may receive higher gain. Further diagnostics are available in the [feature-importance analysis](../../data/benchmarks/feature_importance/README.md).


## Application to a game state


The following example evaluates the installed residual-risk model on a demonstration tracking state. The small spatial grids are used for computational convenience and are not the production-resolution configuration. The reported probability is conditional on successful completion of the current one-second carry; no future state is observed or supplied.


In [ ]:
import pandas as pd
from PRESS.model import ActionSelectionModel, ResidualRiskModel
from PRESS.datasets import load_demo_match
from PRESS import PRESS
from PRESS.extras import BCOS

tracking, possessions, actions, xt = load_demo_match()
press = PRESS(tracking, possessions_data=possessions, actions_data=actions)
frame = 5
area = float(BCOS(tracking).compute_bcos(frame, r_max=8, step=4)["BCOS area"])
result = press.compute_frame(frame, xt, bcos_value=area, ppcf_kwargs={"r_max": 12, "step": 6})
state, context = press.get_state(frame), press.get_frame_context(frame)
action_model = ActionSelectionModel.pretrained()
residual_model = ResidualRiskModel.pretrained()
field = result["PPCF"]
elapsed = 0.5  # demonstration state: time since the start of individual control

record = action_model.build_feature_record(
    x_bc=state["x_bc"], v_bc=state["v_bc"],
    r_atts=state["r_atts"], r_defs=state["r_defs"],
    attacking_side=state["attacking_side"], targets=field["grid"],
    transition_probabilities=field["Transition probabilities"],
    pass_success_probability=field["Pass probability"], bcos=area,
    ball_drive_loss=result["PI"]["Ball drive loss probability"],
    start_type_raw=context["start_type"],
    previous_action_raw=context["previous_action"],
    match_clock_min=tracking.loc[tracking.frame.eq(frame), "match_clock_min"].iloc[0],
    score_difference_prefix=0., completed_canonical_drives=0,
    pitch_length=state["pitch_length"], pitch_width=state["pitch_width"]
)
features = residual_model.features_from_action_record(
    record, time_since_start=elapsed, action_model=action_model
)
loss_probability = float(residual_model.predict_features(features)[0])
print("Terminal loss after a successful one-second carry:", loss_probability)
print("Terminal outcome without loss:", 1 - loss_probability)
